### ingest_daily_support_tickets

In [1]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import os
result = load_dotenv(r"C:\Users\Admin\support-tickets\sample.env")

print(result)

True


In [2]:
os.getenv('REGION')

'eu-north-1'

In [3]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

# ---------- CONFIG ----------
db_config = {
    "host": "127.0.0.1",
    "port": "3306",
    "user": "root",
    "password": "Amit@1234567890",
    "database": "careplus_support_db"
}

S3_BUCKET = "careplusdata9503730654-427532184557-eu-north-1-an"
S3_PREFIX = "Support_Tickets/raw/"
DATE_TRACKER_FILE = "date_tracker.txt"

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": "eu-north-1"
}


In [9]:
def get_engine(config):
    user = config["user"]
    password = quote_plus(config["password"])  # Handles @ and special chars
    host = config["host"]
    port = config["port"]
    database = config["database"]

    connection_string = (
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )

    return create_engine(connection_string)


def upload_to_s3(df, bucket, key):
    csv_buffer = StringIO()

    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)

    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=csv_buffer.getvalue()
    )

    print(f"✅ Uploaded to s3://{bucket}/{key}")


def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()

    return "2025-06-30"


def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)


def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")

    next_date = last_date + timedelta(days=1)

    return next_date.strftime("%Y-%m-%d")


# ---------- MAIN INGESTION LOGIC ----------

def run_ingestion():

    # Connect MySQL
    engine = get_engine(db_config)

    print("✅ Connected to MySQL")

    # Get next date
    last_date = read_last_date(DATE_TRACKER_FILE)

    next_date = get_next_date(last_date)

    print(f"📅 Fetching data for: {next_date}")

    # SQL Query
    query = f"""
        SELECT *
        FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """

    # Read data
    df = pd.read_sql(query, engine)

    print("✅ Data fetched successfully")
    print(df.shape)
    print(df.head())

    # If no data
    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Create S3 file name
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"

    # Upload to S3
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update tracker file
    update_last_date(DATE_TRACKER_FILE, next_date)

    print(f"📅 Updated tracker to {next_date}")

    print("✅ Ingestion Pipeline Completed Successfully")


# ---------- RUN ----------

if __name__ == "__main__":
    run_ingestion()
    

✅ Connected to MySQL
📅 Fetching data for: 2025-07-08
✅ Data fetched successfully
(31, 10)
    ticket_id        created_at       resolved_at   agent priority  \
0  TCK0708000  2025-07-08 00:02  2025-07-09 02:30   Rohit      Low   
1  TCK0708001  2025-07-08 00:41  2025-07-08 17:24   Kavya     High   
2  TCK0708002  2025-07-08 01:13  2025-07-08 10:59   Rohit   Medium   
3  TCK0708003  2025-07-08 01:41              None   Arjun   Medium   
4  TCK0708004  2025-07-08 02:05  2025-07-08 22:01  Ananya       Lw   

  num_interactions         IssUeCat   channel    status agent_feedback  
0                4   Account Locked  Web Form  Resolved                 
1                8   Account Locked  Web Form  Resolved                 
2                8   Account Locked     Email  Resolved                 
3                5  Payment Failure  Web Form      Open                 
4                5       Bug Report      Chat  Resolved                 
✅ Uploaded to s3://careplusdata9503730654-427532184